# Week 4 Exercise - Prompt Sensitivity CodeGen Evaluator

A model-evaluation dashboard for code generation with prompt-style comparison, live progress, run history, and downloadable reports.

In [1]:
# optional installs
# !uv pip install -q openai python-dotenv gradio pandas

In [2]:
import os
import re
import json
import uuid
import sqlite3
import tempfile
import subprocess
from datetime import datetime, timezone
from statistics import mean

import gradio as gr
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv(override=True)

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

DB_PATH = "week4_eval_history.db"

MODELS = [
    {"label": "OpenRouter GPT-4o Mini", "provider": "openrouter", "model": "openai/gpt-4o-mini"},
    {"label": "OpenRouter Llama 3.3 70B", "provider": "openrouter", "model": "meta-llama/llama-3.3-70b-instruct"},
    {"label": "Ollama Llama 3.2", "provider": "ollama", "model": "llama3.2"}
]

PROMPT_STYLES = {
    "concise": "Write only Python code for this task. Return code only.",
    "detailed": "Write Python code only. Include exact function signature and edge-case handling. Return code only.",
    "constraint_driven": "Return only valid Python code, no prose, no markdown, no prints, no input reading."
}

TASKS = [
    {
        "id": "two_sum",
        "prompt": "Implement two_sum(nums, target) that returns indices of two numbers adding to target.",
        "tests": [
            "assert sorted(two_sum([2,7,11,15], 9)) == [0,1]",
            "assert sorted(two_sum([3,2,4], 6)) == [1,2]",
            "assert sorted(two_sum([-1,-2,-3,-4,-5], -8)) == [2,4]"
        ],
        "bench_expr": "two_sum(list(range(10000)), 19997)",
        "bench_loops": 300
    },
    {
        "id": "is_palindrome",
        "prompt": "Implement is_palindrome(s) that ignores non-alphanumeric chars and case.",
        "tests": [
            "assert is_palindrome('A man, a plan, a canal: Panama') is True",
            "assert is_palindrome('race a car') is False",
            "assert is_palindrome(' ') is True"
        ],
        "bench_expr": "is_palindrome('A'*10000 + 'b' + 'A'*10000)",
        "bench_loops": 300
    },
    {
        "id": "group_anagrams",
        "prompt": "Implement group_anagrams(words) to group anagrams together and return list of groups.",
        "tests": [
            "out = group_anagrams(['eat','tea','tan','ate','nat','bat'])",
            "norm = sorted([sorted(g) for g in out])",
            "assert norm == sorted([sorted(['eat','tea','ate']), sorted(['tan','nat']), sorted(['bat'])])"
        ],
        "bench_expr": "group_anagrams(['eat','tea','ate','tan','nat','bat']*200)",
        "bench_loops": 100
    }
]

In [4]:
def init_history_db():
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS run_results (
                run_id TEXT NOT NULL,
                created_at TEXT NOT NULL,
                model_label TEXT NOT NULL,
                prompt_style TEXT NOT NULL,
                task_id TEXT NOT NULL,
                passed INTEGER NOT NULL,
                total INTEGER NOT NULL,
                runtime_ms REAL,
                error_text TEXT
            )
        """)
        conn.commit()

def save_run_rows(run_id, rows):
    now = datetime.now(timezone.utc).isoformat()
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.executemany("""
            INSERT INTO run_results (
                run_id, created_at, model_label, prompt_style, task_id,
                passed, total, runtime_ms, error_text
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            (
                run_id, now, r["model"], r["prompt_style"], r["task_id"],
                int(r["passed"]), int(r["total"]),
                float(r["runtime_ms"]) if r["runtime_ms"] is not None else None,
                r.get("error", "")
            )
            for r in rows
        ])
        conn.commit()

def compare_with_previous_run(current_run_id, current_summary_df):
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute("""
            SELECT run_id FROM run_results
            WHERE run_id != ?
            GROUP BY run_id
            ORDER BY MAX(created_at) DESC
            LIMIT 1
        """, (current_run_id,))
        row = cur.fetchone()

    if not row:
        return "No previous run available for comparison."

    prev_run_id = row[0]
    with sqlite3.connect(DB_PATH) as conn:
        prev_df = pd.read_sql_query(
            "SELECT * FROM run_results WHERE run_id = ?",
            conn,
            params=(prev_run_id,)
        )

    prev_group = prev_df.groupby(["model_label", "prompt_style"], as_index=False).agg(
        passed=("passed", "sum"),
        total=("total", "sum"),
        avg_runtime_ms=("runtime_ms", "mean")
    )
    prev_group["pass_rate"] = prev_group["passed"] / prev_group["total"]

    curr_best = current_summary_df.sort_values(["pass_rate", "avg_runtime_ms"], ascending=[False, True]).iloc[0]
    prev_best = prev_group.sort_values(["pass_rate", "avg_runtime_ms"], ascending=[False, True]).iloc[0]

    curr_avg_pass = float(current_summary_df["pass_rate"].mean())
    prev_avg_pass = float(prev_group["pass_rate"].mean())

    msg = []
    msg.append(f"Compared with previous run: {prev_run_id}")
    msg.append(f"Average pass rate: current={curr_avg_pass:.3f} vs previous={prev_avg_pass:.3f}")
    msg.append(
        "Best combo now: "
        + f"{curr_best['model']} + {curr_best['prompt_style']} (pass_rate={curr_best['pass_rate']:.3f}, runtime={curr_best['avg_runtime_ms']:.3f} ms)"
    )
    msg.append(
        "Best combo previous: "
        + f"{prev_best['model_label']} + {prev_best['prompt_style']} (pass_rate={prev_best['pass_rate']:.3f}, runtime={prev_best['avg_runtime_ms']:.3f} ms)"
    )
    return "\n".join(msg)

init_history_db()

In [5]:
def extract_python_code(text):
    if not text:
        return ""
    match = re.search(r"```(?:python)?\n([\s\S]*?)```", text)
    return match.group(1).strip() if match else text.strip()

def generate_code(model_info, task_prompt, style_prompt):
    client = openrouter_client if model_info["provider"] == "openrouter" else ollama_client
    response = client.chat.completions.create(
        model=model_info["model"],
        messages=[
            {"role": "system", "content": style_prompt},
            {"role": "user", "content": task_prompt}
        ],
        temperature=0.1
    )
    return extract_python_code(response.choices[0].message.content)

def run_tests_and_bench(code, task, timeout_seconds=15):
    runner = f'''
import json, time
{code}
RESULT = {{"passed": 0, "total": {len(task['tests'])}, "error": "", "runtime_ms": None}}

def _run():
    tests = {json.dumps(task['tests'])}
    passed = 0
    for t in tests:
        exec(t, globals(), globals())
        passed += 1

    loops = {task['bench_loops']}
    start = time.perf_counter()
    for _ in range(loops):
        {task['bench_expr']}
    end = time.perf_counter()

    RESULT["passed"] = passed
    RESULT["total"] = len(tests)
    RESULT["runtime_ms"] = ((end - start) / loops) * 1000.0

try:
    _run()
except Exception as e:
    RESULT["error"] = str(e)

print(json.dumps(RESULT))
'''

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(runner)
        path = f.name

    try:
        p = subprocess.run(["python", path], capture_output=True, text=True, timeout=timeout_seconds)
        if p.returncode != 0:
            return {"passed": 0, "total": len(task['tests']), "runtime_ms": None, "error": p.stderr.strip() or "Execution failed"}
        line = p.stdout.strip().splitlines()[-1]
        return json.loads(line)
    except subprocess.TimeoutExpired:
        return {"passed": 0, "total": len(task['tests']), "runtime_ms": None, "error": "Timeout"}
    finally:
        try:
            os.remove(path)
        except OSError:
            pass

def build_artifacts(run_id, summary_df, raw_df, compare_text):
    out_dir = tempfile.mkdtemp(prefix=f"week4_eval_{run_id[:8]}_")
    csv_path = os.path.join(out_dir, "leaderboard.csv")
    json_path = os.path.join(out_dir, "results.json")
    md_path = os.path.join(out_dir, "report.md")

    summary_df.to_csv(csv_path, index=False)

    payload = {
        "run_id": run_id,
        "summary": summary_df.to_dict(orient="records"),
        "raw": raw_df.to_dict(orient="records"),
        "comparison": compare_text
    }
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    try:
        leaderboard_md = summary_df.to_markdown(index=False)
    except Exception:
        leaderboard_md = summary_df.to_string(index=False)

    md = [
        f"# CodeGen Evaluation Report ({run_id})",
        "",
        "## Leaderboard",
        leaderboard_md,
        "",
        "## Comparison With Previous Run",
        compare_text
    ]
    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md))

    return csv_path, json_path, md_path

In [6]:
def run_dashboard(selected_models, selected_styles, task_count, timeout_seconds):
    selected_models = selected_models or []
    selected_styles = selected_styles or []

    if not selected_models:
        yield "Select at least one model.", pd.DataFrame(), pd.DataFrame(), "", "", None, None, None
        return
    if not selected_styles:
        yield "Select at least one prompt style.", pd.DataFrame(), pd.DataFrame(), "", "", None, None, None
        return

    model_map = {m['label']: m for m in MODELS}
    chosen_models = [model_map[m] for m in selected_models]
    chosen_tasks = TASKS[:int(task_count)]

    total_jobs = len(chosen_models) * len(selected_styles) * len(chosen_tasks)
    done = 0

    raw_rows = []
    progress_lines = []

    yield "Starting evaluation...", pd.DataFrame(), pd.DataFrame(), "", "", None, None, None

    for model_info in chosen_models:
        for style_name in selected_styles:
            style_prompt = PROMPT_STYLES[style_name]
            for task in chosen_tasks:
                done += 1
                status = f"[{done}/{total_jobs}] {model_info['label']} | {style_name} | {task['id']}"
                try:
                    code = generate_code(model_info, task['prompt'], style_prompt)
                    result = run_tests_and_bench(code, task, timeout_seconds=int(timeout_seconds))
                except Exception as e:
                    result = {"passed": 0, "total": len(task['tests']), "runtime_ms": None, "error": str(e)}

                raw_rows.append({
                    "model": model_info['label'],
                    "prompt_style": style_name,
                    "task_id": task['id'],
                    "passed": int(result.get('passed', 0)),
                    "total": int(result.get('total', 0)),
                    "runtime_ms": result.get('runtime_ms'),
                    "error": result.get('error', '')
                })

                msg = status + f" -> passed {result.get('passed',0)}/{result.get('total',0)}"
                if result.get('error'):
                    msg += f" | error: {result['error']}"
                progress_lines.append(msg)

                yield "\n".join(progress_lines[-25:]), pd.DataFrame(), pd.DataFrame(), "", "", None, None, None

    raw_df = pd.DataFrame(raw_rows)
    summary_df = raw_df.groupby(["model", "prompt_style"], as_index=False).agg(
        passed=("passed", "sum"),
        total=("total", "sum"),
        avg_runtime_ms=("runtime_ms", "mean")
    )
    summary_df["pass_rate"] = summary_df["passed"] / summary_df["total"]
    summary_df = summary_df.sort_values(["pass_rate", "avg_runtime_ms"], ascending=[False, True]).reset_index(drop=True)

    heat_df = summary_df.pivot(index="model", columns="prompt_style", values="pass_rate").reset_index()

    error_rows = raw_df[raw_df["error"].astype(str) != ""][ ["model", "prompt_style", "task_id", "error"] ]
    if error_rows.empty:
        error_log = "No task failures."
    else:
        lines = [f"{r.model} | {r.prompt_style} | {r.task_id} -> {r.error}" for r in error_rows.itertuples(index=False)]
        error_log = "\n".join(lines)

    run_id = str(uuid.uuid4())
    save_run_rows(run_id, raw_rows)
    compare_text = compare_with_previous_run(run_id, summary_df)
    csv_path, json_path, md_path = build_artifacts(run_id, summary_df, raw_df, compare_text)

    final_progress = "\n".join(progress_lines[-25:] + [f"Done. Run ID: {run_id}"])
    yield final_progress, summary_df, heat_df, error_log, compare_text, csv_path, json_path, md_path

In [ ]:
with gr.Blocks(title="CodeGen Evaluator Dashboard") as app:
    gr.Markdown("# CodeGen Evaluator Dashboard")
    gr.Markdown("Evaluate code-generation models across prompt styles with live progress, history comparison, and downloadable artifacts.")

    with gr.Row():
        models_input = gr.CheckboxGroup(
            choices=[m['label'] for m in MODELS],
            value=[m['label'] for m in MODELS],
            label="Models"
        )
        styles_input = gr.CheckboxGroup(
            choices=list(PROMPT_STYLES.keys()),
            value=list(PROMPT_STYLES.keys()),
            label="Prompt Styles"
        )

    with gr.Row():
        task_count_input = gr.Slider(minimum=1, maximum=len(TASKS), value=len(TASKS), step=1, label="Number of benchmark tasks")
        timeout_input = gr.Slider(minimum=5, maximum=30, value=15, step=1, label="Test timeout (seconds)")

    run_btn = gr.Button("Run Evaluation", variant="primary")

    progress_output = gr.Textbox(label="Live Progress", lines=10)
    leaderboard_output = gr.Dataframe(label="Leaderboard")
    heatmap_output = gr.Dataframe(label="Prompt-Style Heatmap (pass_rate)")
    error_output = gr.Textbox(label="Per-Task Failure Log", lines=8)
    compare_output = gr.Textbox(label="Comparison With Previous Run", lines=6)

    csv_file = gr.File(label="Download CSV")
    json_file = gr.File(label="Download JSON")
    md_file = gr.File(label="Download Markdown Report")

    run_btn.click(
        fn=run_dashboard,
        inputs=[models_input, styles_input, task_count_input, timeout_input],
        outputs=[progress_output, leaderboard_output, heatmap_output, error_output, compare_output, csv_file, json_file, md_file]
    )

app.launch()